In [ ]:
import rdkit.Chem as Chem
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

smiles = 'CC1=C(C(=O)N2CCCCC2=N1)CCN3CCC(CC3)C4=NOC5=C4C=CC(=C5)F'
Chem.MolFromSmiles(smiles)


In [ ]:
from dockstring import load_target

target = load_target('DRD2')
score, aux = target.dock(smiles)
print(score)

In [ ]:
def dock_wrapper(mols):
    """
    function that docks every mol in array mols using dockstring.
    hard filter at 30 heavy atoms to avoid model exploiting docking score.
    """
    scores = []
    for mol in mols:
        if len(mol.GetAtoms())<=30:
            smi = Chem.MolToSmiles(mol)
            score, aux = target.dock(smi)
            scores.append(score)
        else:
            scores.append(0.000)
    return scores

dock_wrapper([Chem.MolFromSmiles("CCCC"),Chem.MolFromSmiles("c1ccccc1C"),Chem.MolFromSmiles("c1ccccc1C"*6)])

In [ ]:
from lacan import lacan,gen
from lacan.gen import GAReporter
p = lacan.load_profile("chembl")

reporter = GAReporter(label="docking")
winners_shape = gen.generate_optimized_molecules(
    dock_wrapper, p,
    preset="docking",
    # scoring_budget=20,    # molecules scored per generation (preset default)
    # startN=20,            # initial population size
    generations=15,
    higher_is_better=False,
    n_jobs=12,
    quiet=False,
    seed=888,
    callback=reporter
)
allc_shape = sorted(winners_shape, key=lambda x: -x[1])
print(f"\n{len(allc_shape)} molecules in HallOfFame")
d = Draw.MolsToGridImage(
    [Chem.MolFromSmiles(c[0]) for c in allc_shape],
    useSVG=True, molsPerRow=4,
    legends=[str(round(c[1], 3)) for c in allc_shape],
    maxMols=20,
)
display(d)

In [ ]:
from rdkit.Chem import Draw
allc_shape = sorted(winners_shape, key=lambda x: x[1])
print(f"\n{len(allc_shape)} molecules in HallOfFame")
d = Draw.MolsToGridImage(
    [Chem.MolFromSmiles(c[0]) for c in allc_shape],
    useSVG=True, molsPerRow=4,
    legends=[str(round(c[1], 3)) for c in allc_shape],
    maxMols=8,
)
display(d)

In [ ]:
reporter.label = "DRD2 Docking"
reporter.plot()

In [ ]:
aux["ligand"]

In [ ]:
score2, aux2 = target.dock(allc_shape[0][0])

In [ ]:
score3, aux3 = target.dock(allc_shape[1][0])

In [ ]:
score4, aux4 = target.dock(allc_shape[2][0])

In [ ]:
import py3Dmol
from rdkit.Chem import MolToMolBlock


view = py3Dmol.view(width=700, height=500)

view.addModel(MolToMolBlock(aux["ligand"]), 'mol')
view.setStyle({'model': 0}, {'stick': { 'radius': 0.12,'colorscheme': 'cyanCarbon'}})

view.addModel(MolToMolBlock(aux2["ligand"]), 'mol')
view.setStyle({'model': 1}, {'stick': {'radius': 0.12,'colorscheme': 'purpleCarbon'}})

"""view.addModel(MolToMolBlock(aux3["ligand"]), 'mol')
view.setStyle({'model': 2}, {'stick': {'radius': 0.12,'colorscheme': 'yellowCarbon'}})

view.addModel(MolToMolBlock(aux4["ligand"]), 'mol')
view.setStyle({'model': 3}, {'stick': {'radius': 0.12,'colorscheme': 'orangeCarbon'}})"""


view.zoomTo()
view.show()

In [ ]:
wri = Chem.SDWriter("D2_docking_poses.sdf")
for a in [aux,aux2,aux3,aux4]:
    wri.write(a["ligand"])
wri.close()